# AR-SSL4M Pretraining with Mixed Data Sources (Corrected)

This notebook handles the setup and pretraining of the AR-SSL4M model using:
- LIDC data from Google Drive
- BraTS data from Google Cloud Storage 
- DeepLesion data from Google Drive

In [ ]:
# Mount Google Drive and authenticate GCS
from google.colab import drive
from google.colab import auth

drive.mount('/content/drive')
auth.authenticate_user()  # 验证 GCS 权限

In [ ]:
# Configure data sources (CORRECTED)
import os

# 项目 ID 和数据源配置
PROJECT_ID = 'brats-preprocess'
GCS_BRATS_BUCKET = "gs://brats_preprocessed_npy"  # BraTS 数据在 GCS
LIDC_PATH = '/content/drive/MyDrive/dataset/LIDC-IDRI/patch_random_spatial'  # LIDC 在 Drive
DEEPLESION_PATH = '/content/drive/MyDrive/dataset/pretrain/DeepLesion/npy'  # DeepLesion 在 Drive

# 设置 GCP 项目
!gcloud config set project {PROJECT_ID}

# 检查 LIDC 数据 (Drive)
if os.path.exists(LIDC_PATH):
    lidc_files = [f for f in os.listdir(LIDC_PATH) if f.endswith('.npy')]
    print(f"LIDC found: {len(lidc_files)} files at {LIDC_PATH}")
else:
    print(f"LIDC NOT found at {LIDC_PATH}")

# 检查 DeepLesion 数据 (Drive)
if os.path.exists(DEEPLESION_PATH):
    deeplesion_files = [f for f in os.listdir(DEEPLESION_PATH) if f.endswith('.npy')]
    print(f"DeepLesion found: {len(deeplesion_files)} files at {DEEPLESION_PATH}")
else:
    print(f"DeepLesion NOT found at {DEEPLESION_PATH}")

# 检查 BraTS 数据 (GCS)
print(f"\nChecking BraTS data in GCS...")
!gsutil ls {GCS_BRATS_BUCKET} | head -10
print(f"BraTS data configured from: {GCS_BRATS_BUCKET}")

In [ ]:
# Generate Training Lists for Mixed Data Sources (CORRECTED)
# LIDC (Drive) + BraTS (GCS) + DeepLesion (Drive)

import os
import numpy as np
from tqdm import tqdm

# 本地临时目录用于存储训练列表
LOCAL_TEMP_ROOT = "/content/temp_lists"
os.makedirs(LOCAL_TEMP_ROOT, exist_ok=True)

# === 1. 生成 LIDC Spatial 列表 (从 Drive) ===
spatial_list_path = os.path.join(LOCAL_TEMP_ROOT, 'train_spatial.txt')
print("Generating LIDC spatial list from Drive...")

if os.path.exists(LIDC_PATH):
    lidc_files = []
    npy_files = [f for f in os.listdir(LIDC_PATH) if f.endswith('.npy')]
    
    print(f"Checking {len(npy_files)} LIDC files...")
    for f in tqdm(npy_files):
        full_path = os.path.join(LIDC_PATH, f)
        try:
            # 快速检查文件是否有效
            data = np.load(full_path, mmap_mode='r')
            if data.size > 0:
                lidc_files.append(full_path)
        except Exception as e:
            print(f"Skipping corrupted LIDC file: {f}")
    
    with open(spatial_list_path, 'w') as f:
        f.write('\n'.join(lidc_files))
    print(f"Found {len(lidc_files)} valid LIDC files")
else:
    print(f"LIDC path not found: {LIDC_PATH}")
    with open(spatial_list_path, 'w') as f:
        f.write('')

# === 2. 生成 BraTS Contrast 列表 (从 GCS) ===
contrast_list_path = os.path.join(LOCAL_TEMP_ROOT, 'train_contrast.txt')
print("\nGenerating BraTS contrast list from GCS...")

# 获取 GCS 中所有的 .npy 文件
print("Scanning GCS bucket for BraTS files...")
!gsutil ls -r {GCS_BRATS_BUCKET}/**/*.npy > /tmp/all_brats_files.txt 2>/dev/null || echo "No .npy files found" > /tmp/all_brats_files.txt

# 读取所有文件列表
try:
    with open('/tmp/all_brats_files.txt', 'r') as f:
        all_files = [line.strip() for line in f if line.strip().endswith('.npy')]
    print(f"Found {len(all_files)} total .npy files in GCS")
except FileNotFoundError:
    print("No files found in GCS bucket. Creating empty contrast list.")
    all_files = []

valid_brats_cases = []
if all_files:
    # 提取所有 t1n 文件作为基准
    t1n_files = [f for f in all_files if '.t1n.npy' in f]
    print(f"Found {len(t1n_files)} t1n files")
    
    if t1n_files:
        print(f"Sample t1n files: {t1n_files[:3]}")
        
        # 验证每个 t1n 文件是否有对应的 t1c, t2w, t2f 文件
        all_files_set = set(all_files)
        
        for t1n_file in t1n_files:
            # 获取 base name (去掉 .t1n.npy)
            base = t1n_file.replace('.t1n.npy', '')
            
            # 检查是否存在所有 4 个模态
            required_files = [
                f"{base}.t1n.npy",
                f"{base}.t1c.npy", 
                f"{base}.t2w.npy",
                f"{base}.t2f.npy"
            ]
            
            # 检查所有必需文件是否都存在
            if all(req_file in all_files_set for req_file in required_files):
                valid_brats_cases.append(base)  # 存储 base name
    
    print(f"Found {len(valid_brats_cases)} valid BraTS contrast cases (with all 4 modalities)")
else:
    print("No BraTS files found in GCS")

# 写入 contrast 列表
with open(contrast_list_path, 'w') as f:
    f.write('\n'.join(valid_brats_cases))

if len(valid_brats_cases) == 0:
    print("WARNING: No valid BraTS cases found!")
else:
    print(f"BraTS contrast list saved to: {contrast_list_path}")

# === 3. 生成 DeepLesion Semantic 列表 (从 Drive) ===
semantic_list_path = os.path.join(LOCAL_TEMP_ROOT, 'train_semantic.txt')
if os.path.exists(DEEPLESION_PATH):
    print("\nGenerating DeepLesion semantic list from Drive...")
    deeplesion_files = []
    npy_files = [f for f in os.listdir(DEEPLESION_PATH) if f.endswith('.npy')]
    
    print(f"Checking {len(npy_files)} DeepLesion files...")
    for f in tqdm(npy_files):
        full_path = os.path.join(DEEPLESION_PATH, f)
        try:
            # 快速检查文件是否有效
            data = np.load(full_path, mmap_mode='r')
            if data.size > 0:
                deeplesion_files.append(full_path)
        except Exception as e:
            print(f"Skipping corrupted file: {f}")
    
    with open(semantic_list_path, 'w') as f:
        f.write('\n'.join(deeplesion_files))
    print(f"Found {len(deeplesion_files)} valid DeepLesion files")
else:
    print(f"\nDeepLesion path not found: {DEEPLESION_PATH}")
    # 创建空的语义列表
    with open(semantic_list_path, 'w') as f:
        f.write('')
    print("Created empty semantic list")

print(f"\n=== Training Lists Generated ===")
print(f"Spatial (LIDC): {spatial_list_path}")
print(f"Contrast (BraTS): {contrast_list_path}")
print(f"Semantic (DeepLesion): {semantic_list_path}")